In [1]:
import pandas as pd
import numpy as np

# ============================================================
# 1. File paths
# ============================================================

true_path = "3000testbank.csv"
pred_path = "GNN_3000testbank_predictions_renamed.csv"

true_df = pd.read_csv(true_path)
pred_df = pd.read_csv(pred_path)

print("Loaded files:")
print("True file shape:", true_df.shape)
print("Predicted file shape:", pred_df.shape)

Loaded files:
True file shape: (3000, 12)
Predicted file shape: (3000, 11)


In [2]:
# ============================================================
# 2. Columns to percentile normalize
# ============================================================
# Only these columns will be converted to percentile scores.
# aromaticity, molecular_weight, and log_P will not be changed.

property_cols = [
    "sigma_780nm",
    "max_sigma",
    "ISC(S1-T1)",
    "Solubility",
    "BoilingPoint",
    "SAscore",
    "Tox_score",
]

# Check required columns
missing_true = [col for col in property_cols if col not in true_df.columns]
missing_pred = [col for col in property_cols if col not in pred_df.columns]

if missing_true:
    raise KeyError(f"Missing columns in true file: {missing_true}")

if missing_pred:
    raise KeyError(f"Missing columns in predicted file: {missing_pred}")

In [3]:
# ============================================================
# 3. Percentile normalization function
# ============================================================
# Formula:
# P = ((Rank(x_i) - 0.5) / N) * 100
#
# Predicted values are normalized using the true dataset distribution
# as the reference, so true and predicted values are comparable.

def percentile_normalize_using_reference(values, reference):
    """
    Convert values to percentile-normalized scores using a reference distribution.

    Blank / invalid cells stay NaN.

    Percentile formula:
        P = ((rank - 0.5) / N) * 100
    """

    values = pd.to_numeric(values, errors="coerce")
    reference = pd.to_numeric(reference, errors="coerce").dropna()

    result = pd.Series(np.nan, index=values.index)

    if len(reference) == 0:
        return result

    ref_sorted = np.sort(reference.to_numpy())
    N = len(ref_sorted)

    percentile_positions = ((np.arange(1, N + 1) - 0.5) / N) * 100

    ref_table = pd.DataFrame({
        "value": ref_sorted,
        "percentile": percentile_positions,
    })

    # Handle duplicate raw values by averaging their percentile positions
    ref_unique = ref_table.groupby("value", as_index=False)["percentile"].mean()

    valid = values.notna()

    result.loc[valid] = np.interp(
        values.loc[valid],
        ref_unique["value"],
        ref_unique["percentile"],
        left=ref_unique["percentile"].iloc[0],
        right=ref_unique["percentile"].iloc[-1],
    )

    return result


In [4]:
# ============================================================
# 4. Dump normalized score files
# ============================================================
# This directly replaces the selected property columns with
# percentile-normalized values.
#
# Other columns, including aromaticity, molecular_weight, and log_P,
# are left unchanged.

true_norm = true_df.copy()
pred_norm = pred_df.copy()

for col in property_cols:
    true_norm[col] = percentile_normalize_using_reference(
        true_df[col],
        true_df[col]
    )

    pred_norm[col] = percentile_normalize_using_reference(
        pred_df[col],
        true_df[col]
    )

true_norm_path = "true_normalized_scores.csv"
pred_norm_path = "predicted_normalized_scores.csv"

true_norm.to_csv(true_norm_path, index=False)
pred_norm.to_csv(pred_norm_path, index=False)

print("\nSaved normalized score files:")
print(true_norm_path)
print(pred_norm_path)


Saved normalized score files:
true_normalized_scores.csv
predicted_normalized_scores.csv


In [5]:
# ============================================================
# 5. Define positive and negative weights
# ============================================================
# Positive weight: higher percentile is better.
# Negative weight: lower percentile is better.
#
# ISC(S1-T1): lower is better, so negative.
# Tox_score: lower toxicity is better, so negative.
#
# SA_score: 1 means easy to synthesize, so higher is better.

weights = {
    "sigma_780nm": 0.30,
    "max_sigma": 0.15,
    "ISC(S1-T1)": -0.20,
    "Solubility": 0.10,
    "BoilingPoint": 0.05,
    "SAscore": 0.10,
    "Tox_score": -0.10,
}

# Check that all weighted columns exist
missing_weight_cols_true = [col for col in weights if col not in true_norm.columns]
missing_weight_cols_pred = [col for col in weights if col not in pred_norm.columns]

if missing_weight_cols_true:
    raise KeyError(f"Missing weighted columns in true normalized file: {missing_weight_cols_true}")

if missing_weight_cols_pred:
    raise KeyError(f"Missing weighted columns in predicted normalized file: {missing_weight_cols_pred}")


In [6]:
# ============================================================
# 6. Final score calculation
# ============================================================
# Direct formula:
#
# final_score =
# 0.30 * sigma_780nm
# + 0.15 * max_sigma
# - 0.20 * ISC(S1-T1)
# + 0.10 * CrystalSolubility
# + 0.05 * BoilingPoint
# + 0.10 * SA_score
# - 0.10 * Tox_score
#
# Missing values are ignored.
# The score is divided by the available absolute weight.
# Rows with fewer than min_valid properties get NaN final_score.

def calculate_final_score(df, weights, min_valid=5):
    df = df.copy()

    weighted_sum = pd.Series(0.0, index=df.index)
    available_abs_weight = pd.Series(0.0, index=df.index)
    valid_count = pd.Series(0, index=df.index)

    for col, weight in weights.items():
        values = pd.to_numeric(df[col], errors="coerce")
        valid = values.notna()

        weighted_sum.loc[valid] += values.loc[valid] * weight
        available_abs_weight.loc[valid] += abs(weight)
        valid_count.loc[valid] += 1

    df["valid_property_count"] = valid_count
    df["coverage"] = valid_count / len(weights)

    df["final_score"] = weighted_sum / available_abs_weight
    df.loc[available_abs_weight == 0, "final_score"] = np.nan

    df["score_reliable"] = df["valid_property_count"] >= min_valid
    df.loc[df["valid_property_count"] < min_valid, "final_score"] = np.nan

    return df

In [7]:
# ============================================================
# 7. Dump final score files
# ============================================================

true_scored = calculate_final_score(
    true_norm,
    weights,
    min_valid=5
)

pred_scored = calculate_final_score(
    pred_norm,
    weights,
    min_valid=5
)

true_scored_path = "true_final_scores.csv"
pred_scored_path = "predicted_final_scores.csv"

true_scored.to_csv(true_scored_path, index=False)
pred_scored.to_csv(pred_scored_path, index=False)

print("\nSaved final score files:")
print(true_scored_path)
print(pred_scored_path)


Saved final score files:
true_final_scores.csv
predicted_final_scores.csv


In [8]:
# ============================================================
# 8. Dump final score sorted files
# ============================================================

true_sorted = true_scored.sort_values(
    "final_score",
    ascending=False,
    na_position="last"
)

pred_sorted = pred_scored.sort_values(
    "final_score",
    ascending=False,
    na_position="last"
)

true_sorted_path = "true_final_scores_sorted.csv"
pred_sorted_path = "predicted_final_scores_sorted.csv"

true_sorted.to_csv(true_sorted_path, index=False)
pred_sorted.to_csv(pred_sorted_path, index=False)

print("\nSaved sorted final score files:")
print(true_sorted_path)
print(pred_sorted_path)


Saved sorted final score files:
true_final_scores_sorted.csv
predicted_final_scores_sorted.csv


In [9]:
import pandas as pd
import numpy as np

# ============================================================
# 1. Load final score files
# ============================================================

true_path = "true_final_scores.csv"
pred_path = "predicted_final_scores.csv"

true_df = pd.read_csv(true_path)
pred_df = pd.read_csv(pred_path)

id_col = "SMILES"
score_col = "final_score"
k = 100

# ============================================================
# 2. Make sure score column is numeric
# ============================================================

true_df[score_col] = pd.to_numeric(true_df[score_col], errors="coerce")
pred_df[score_col] = pd.to_numeric(pred_df[score_col], errors="coerce")

# ============================================================
# 3. Sort by final score and take top 100
# ============================================================

true_sorted = true_df.sort_values(score_col, ascending=False, na_position="last")
pred_sorted = pred_df.sort_values(score_col, ascending=False, na_position="last")

true_top_k = true_sorted.head(k)
pred_top_k = pred_sorted.head(k)

In [10]:
# ============================================================
# 4. Convert top 100 molecules to sets
# ============================================================

true_top_k_set = set(true_top_k[id_col])
pred_top_k_set = set(pred_top_k[id_col])

# ============================================================
# 5. Calculate hits
# ============================================================

hits = true_top_k_set.intersection(pred_top_k_set)
hits_at_k = len(hits)

retrieved_at_k = k
total_relevant = k

# ============================================================
# 6. Calculate Precision@100 and Recall@100
# ============================================================

precision_at_k = hits_at_k / retrieved_at_k
recall_at_k = hits_at_k / total_relevant

# ============================================================
# 7. Calculate Enrichment@100
# ============================================================
# Random baseline = fraction of true top-k molecules in the whole dataset
#
# EF@k = Precision@k / baseline_hit_rate
# baseline_hit_rate = k / N

N = len(true_df)

baseline_hit_rate = k / N
enrichment_at_k = precision_at_k / baseline_hit_rate

# ============================================================
# 8. Print results
# ============================================================

print(f"Total molecules N: {N}")
print(f"Retrieved@{k}: {retrieved_at_k}")
print(f"Hits@{k}: {hits_at_k}")
print(f"Precision@{k}: {precision_at_k:.4f}")
print(f"Recall@{k}: {recall_at_k:.4f}")
print(f"Enrichment@{k}: {enrichment_at_k:.4f}")

Total molecules N: 3000
Retrieved@100: 100
Hits@100: 30
Precision@100: 0.3000
Recall@100: 0.3000
Enrichment@100: 9.0000


In [11]:
# ============================================================
# 9. Save metric result
# ============================================================

metrics_df = pd.DataFrame({
    "k": [k],
    "N": [N],
    f"retrieved@{k}": [retrieved_at_k],
    f"hits@{k}": [hits_at_k],
    f"precision@{k}": [precision_at_k],
    f"recall@{k}": [recall_at_k],
    f"enrichment@{k}": [enrichment_at_k],
    "baseline_hit_rate": [baseline_hit_rate],
})

metrics_df.to_csv(f"ranking_metrics_at_{k}.csv", index=False)

metrics_df

,k,N,retrieved@100,hits@100,precision@100,recall@100,enrichment@100,baseline_hit_rate
0,100,3000,100,30,0.3,0.3,9.0,0.033333
